# The findings ledger

The ledger is the product. Everything else in the report — cards, coverage —
exists to support it or to bound what it can honestly say. This chapter is
about reading findings well: what each field promises, how the ledger is
organized and selected, and which claims are safe to act on.

One idea organizes everything else: **a finding is a statement, independent
of the detector that found it.** "quantity is a function of double_quantity"
is a statement; a lattice search, an identity probe, or a future detector
are just different ways of noticing it. The report says what it claims
(category, type, subject, specification) and separately who measured it and
how (the evidence list). Two detectors that notice the same statement end
up as one finding with two evidence records.

Findings are written for two readers at once. A human reads the `claim` —
one precise sentence. A program reads everything else: `category` for
routing, `subject` for scoping, `evidence` for checking, `interestingness`
for ranking. An agent that quotes a finding should quote the claim verbatim
and carry the `id` and evidence alongside it.

## Anatomy of a finding

| Field | Type | What it holds |
|---|---|---|
| `id` | string | Content hash of the statement (`category-type-subject-specification`); stable across reruns and detector changes |
| `category` | string | The taxonomy entry: what kind of statement this is |
| `type` | string? | Sub-type where a category needs one (e.g. `functional`/`predictive` inside `dependencies`) |
| `subject` | dict | What the statement is about: `columns` (always), plus `inputs` and `target` for dependencies, `unit` for keys and duplicates |
| `specification` | dict | The logical form: `form` (e.g. `value_mapping`, `affine`), `conditions`/`outcome` predicates for rules, `definition` details (comparison, missing-value handling, role, calendar reference) |
| `claim` | string | One precise sentence, written to be quoted verbatim |
| `evidence` | list | One record per contributing detector: method, basis, scope, validation, metrics |
| `interestingness` | float | Deterministic score in $[0,1]$ that ranks the ledger |
| `interpretation` | string? | A suggested modeling or data-handling implication, when one exists |

The fixture below is the quickstart's orders table plus a few columns with
known problems — a heavily concentrated channel and invoice numbers written
in two formats — so the ledger has something to say on several fronts at
once.

In [ ]:
import json
import random

import polars as pl

import onceover

rng = random.Random(7)
n = 1500

region_id = [rng.randrange(3) for _ in range(n)]
region_name = [["emea", "amer", "apac"][r] for r in region_id]
order_id = list(range(n))
external_ref = [f"R{i:05}" for i in range(n)]
external_ref[-1] = external_ref[0]
external_ref[-2] = external_ref[1]
quantity = [rng.randrange(1, 11) for _ in range(n)]
double_quantity = [2 * q for q in quantity]
channel = ["web" if rng.random() < 0.99 else "app" for _ in range(n)]
invoice_no = [f"INV-{i:04d}" if i % 2 == 0 else f"{i:04d}" for i in range(n)]

df = pl.DataFrame({
    "order_id": order_id,
    "external_ref": external_ref,
    "region_id": region_id,
    "region_name": region_name,
    "quantity": quantity,
    "double_quantity": double_quantity,
    "channel": channel,
    "invoice_no": invoice_no,
})

report = onceover.profile(df)
print(report.overview["headline"])

## The statement taxonomy

Every finding carries a `category`, and the category is the join key
between the ledger and the rest of the documentation: each methodology
chapter derives the statements its detector emits, and the coverage block
reports detectors whose findings line up with them.

```{admonition} The taxonomy is evolving
:class: warning

The statement taxonomy — categories, types, and specification forms — is
evolving and is likely to change in future releases. Programs that consume
reports should read `schema_version`, match categories and types
defensively, and treat unknown values as opaque rather than fatal. Schema
0.2 itself replaced the flat finding-kind strings of 0.1, breaking
consumers that matched on them.
```

| Category | Type(s) | What the statement says | Detector(s) |
|---|---|---|---|
| `missingness` | — | a column is entirely null | `column_profile` |
| `concentration` | — | a column is constant, nearly constant, or heavily dominated by few values | `column_profile` |
| `column_roles` | — | a numeric column behaves like a category; a column looks like an identifier (inferred, not verified) | `column_profile` |
| `uniqueness` | — | a column or column set identifies rows: primary key, composite key, or near-unique with repeats | `keys` |
| `duplicates` | — | rows repeat earlier rows exactly | `row_duplicates` |
| `formats` | — | a string column mixes value formats | `column_profile` |
| `dependencies` | `functional` | one column is a (near-)function of others: a value mapping, an affine formula, or a piecewise-linear regime | `functional_dependencies`, `identities`, `regimes` |
| `dependencies` | `predictive` | a target is predictable from inputs, with named validation | `gbm`, `decision_tables`, `regimes` |
| `conditional_rules` | `if_then` | when predicates hold, an outcome holds, with measured rate and lift | `prism`, `association_rules` |
| `associations` | `statistical` | numeric columns move together (Pearson, Spearman, ξ) | `numeric_dependence` |
| `gaps` | — | a temporal column is missing expected values, against a disclosed calendar reference | `dates` |
| `intervals` | — | a temporal column has irregular spacing | `dates` |
| `ordering` | — | a column is sorted with row order | `sortedness` |
| `changes` | `residual_level` | a target's residual level shifts after a date | `regimes` |

The current distribution, as a histogram over this table's ledger:

In [ ]:
from collections import Counter

print(Counter((f["category"], f.get("type")) for f in report.findings))

## The evidence list

Every finding carries at least one evidence record, and each record has five
parts:

| Part | Fields | What they hold |
|---|---|---|
| `method` | `detector`, `algorithm`, `provenance` | who found it, with what algorithm and lineage |
| `basis` | `measured` \| `estimated` \| `inferred` | counted or verified directly; computed from a sample or a model; or a rule-of-thumb reading of measurements |
| `scope` | `sampling` (`full`\|`sample`), `population_rows`, `rows_evaluated`, `missing_values` | over how much of the table, and how nulls were treated |
| `validation` | `method`, `training_rows`, `evaluation_rows`, `folds`, `used_for_selection` | whether the numbers come from direct comparison, a holdout, cross-validation, or a diagnostic partition |
| `metrics` | free-form | the numbers backing the claim: violations, rates, coefficients, effect sizes |

`basis` and `validation` together replace the single all-or-nothing
confidence label of older schemas, and they separate two questions that
label conflated: *what kind of measurement is this* (basis) and *was it
checked against data the detector did not train on* (validation). A
functional dependency is `measured` with `direct_comparison`; a
predictive dependency is `estimated` with `diagnostic_holdout`; a
conditional rule is discovered by one detector and then *re-measured* on a
diagnostic partition, so it ends up with two evidence records carrying two
different bases — both honest about what they did.

## One statement, several detectors

Consolidation is easier to see than to describe. On this table, the
predictive statement about `region_id` was noticed twice — by the
predictability screen and by the decision-table search — and the ledger
keeps one statement with both records:

In [ ]:
merged = next(f for f in report.findings
              if f["category"] == "dependencies"
              and len(f["evidence"]) > 1)
print(merged["claim"])
for ev in merged["evidence"]:
    m = ev["method"]
    print(f"  {m['detector']:16s} basis={ev['basis']:9s} "
          f"validation={ev['validation']['method']}")

The two detectors measured different things — a holdout accuracy and a
cross-validated lookup accuracy — and both records survive untouched; the
ledger never averages or pools evidence. Statements merge when their
category, type, subject, and specification match; the claim kept is the
one from the best-evaluated record.

Conditional rules show the same pattern with a sharper edge. PRISM
discovers a rule on data it may have used to grow it; a shared evaluator
then re-counts every rule on a deterministic train/evaluation partition
that selection reuses. One rule, two evidence records:

In [ ]:
rule = next(f for f in report.findings if f["category"] == "conditional_rules")
print(rule["claim"])
for ev in rule["evidence"]:
    print(f"  {ev['method']['detector']:16s} basis={ev['basis']:9s} "
          f"validation={ev['validation']['method']}")
print("  evaluated metrics:", {k: v for k, v in rule["evidence"][-1]["metrics"].items()
                                if k in ("conditional_rate", "baseline_rate", "lift", "joint_count")})

The claim quotes the evaluated numbers, not the discovery numbers —
"holds in 118 of 118 matching diagnostic rows" — and the discovery record
stays attached for provenance.

## Ids and determinism

Finding ids are content hashes of the canonical statement: the category,
type, sorted subject, and canonical specification — never the rank, never
the detector, never the report. Rerunning on the same data reproduces the
same ids; swapping the detector that finds a statement does not change the
id; and two reports from two versions of the package can be diffed by id
where they agree. Ids are not sequential, so do not parse them — treat them
as opaque join keys.

## Ranking: interestingness

Interestingness scores every statement as a weighted mix of four readable
ingredients:

$$\text{interestingness} = 0.45 \cdot \text{quality} + 0.35 \cdot
\text{strength} + 0.10 \cdot \text{specificity} + 0.10 \cdot \text{simplicity}$$

- **quality** — how the evidence was established: 1.0 for direct
  comparison on full data; 0.8 for holdouts, cross-validation, and
  diagnostic partitions; below that, by basis (measured 0.85, estimated
  0.6, inferred 0.4).
- **strength** — per category: for functional dependencies, the share of
  rows that obey the mapping ($1 - \text{violations}/n$); for predictive
  dependencies, held-out skill above baseline; for rules, a blend of
  rate-gain and lift capped by support; for associations, the largest
  measured effect.
- **specificity** — does the statement have a concrete logical form
  (1.0), or is it a bare observation (0.5)?
- **simplicity** — fewer inputs and conditions score higher:
  $1/(1 + (\text{inputs} + \text{conditions})/8)$.

The functional dependency from the quickstart scores 0.9889 exactly:
quality 1.0 (direct comparison, full data), strength 1.0 (zero violations
in 2,000 rows), specificity 1.0 (a `value_mapping` form), simplicity
$1/(1 + 1/8) = 0.889$. The exact decomposition lives in
[](../methodology/evidence.ipynb); what matters when reading is that the
top of the ledger is where verified structure lives, and the score is
reproducible from the evidence.

## Selection: from candidates to a report

A table can produce hundreds of candidate statements; a report carries a
bounded, selected ledger. Selection is deterministic and disclosed:

| Policy knob | Default | Effect |
|---|---:|---|
| `max_findings` | 100 | report-wide cap after ranking |
| `max_rules` | 20 | cap on conditional rules |
| `rules_per_outcome` | 2 | at most this many rules per outcome column |
| `predictive_findings_per_target` | 3 | at most this many predictive statements per target |

The Python API exposes `max_findings` and `max_rules`; the Rust
`SelectionPolicy` exposes the full set. The counts travel with the report —
`report.data["selection"]` records how many candidates, verified
candidates, distinct statements, and selected findings the run produced —
and the `rule_evaluation` coverage entry reports how many rule candidates
passed the common diagnostic partition.

## Groups

`report.groups` links findings that share columns — connected components
over the subjects of dependencies, rules, and associations. On the
quickstart table, `quantity` and `double_quantity` appear in one group
with every statement about them. Membership says "these columns form a
connected story"; it deliberately does **not** say "these columns are
redundant" — that stronger claim depends on direction and form, and the
group description says so.

## Filtering the ledger

The ledger is a plain list of dicts, so anything expressible in Python
works. Four patterns cover most needs.

By category — "give me the structure claims":

In [ ]:
structure = [f for f in report.findings
             if f["category"] in ("uniqueness", "duplicates", "dependencies",
                                  "conditional_rules")]
print(len(structure), "structural findings")
for f in structure[:5]:
    print(f"  [{f['category']}/{f.get('type', '-')}] {f['claim'][:70]}")

By subject column — "what does the report know about `quantity`?". The
subject's `columns` list is the canonical place to look:

In [ ]:
for f in report.findings:
    if "quantity" in f["subject"]["columns"]:
        print(f"[{f['category']}/{f.get('type', '-')}] {f['claim'][:80]}")

By basis — "which statements were directly measured on full data?

In [ ]:
measured = [f for f in report.findings
              if any(ev["basis"] == "measured" and ev["scope"]["sampling"] == "full"
                     for ev in f["evidence"])]
print(len(measured), "statements measured on full data")

And by validation, when the question is "what survived a holdout?" —
look for `diagnostic_holdout`, `cross_validation`, or `diagnostic_partition`
in the evidence.

## What to act on

The `interpretation` field records the implication the profiler is
confident enough to suggest: deduplicate before joining, inspect a mapping
before treating a column as derived, treat a numeric column as
categorical. Interpretations are suggestions, not verdicts; the evidence is
the part built for decision-making.

| Evidence pattern | Safe to… |
|---|---|
| `measured` + `direct_comparison` + `full` sampling | Enforce: use as constraints, join keys, derived-column checks |
| `measured` + `diagnostic_partition` (rules) | Rely on the rate, lift, and support as effect sizes on unseen rows |
| `estimated` + `diagnostic_holdout` / `cross_validation` | Investigate: effect sizes are trustworthy, guarantees are not |
| `inferred` | Consider: a candidate reading (e.g. a role), not a verdict |

```{admonition} Absence is not evidence
:class: warning

A finding that is absent may mean the property is absent, or that the
detector was capped, sampled, skipped, or not selected. The coverage block
distinguishes these — read it before concluding "the profiler found
nothing" means "there is nothing to find". [](coverage.ipynb) explains the
vocabulary.
```

## Choosing detectors

Every part of the ledger so far comes from built-in detectors, and they
are selectable. `onceover.available_detectors()` lists them with the
categories they produce and the detectors they require; `profile(df,
detectors=[...])` selects a subset:

In [ ]:
for d in onceover.available_detectors()[:4]:
    print(f"{d['id']:24s} categories={d['categories']} requires={d['requires']}")

subset = onceover.profile(df, detectors=["keys", "functional_dependencies"])
print(subset.overview["headline"])

Selection has resource semantics: a detector that another requires (say
`gbm`, which `decision_tables` needs for its target shortlists) still runs
when its dependent is selected, but contributes no findings of its own —
its coverage line says "required as a resource; findings not selected".
Deselected detectors appear in coverage as "detector not enabled". This
makes the ledger compositionally honest: whatever is in it was produced by
exactly the detectors you asked for, plus the resources they needed.